# Day 28: Memory & State in Agents

Implement short‑term (conversation buffer) and long‑term (semantic) memory.

In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# We'll also need embeddings for long‑term memory
import faiss
import numpy as np
from typing import List, Dict, Any

## 1. Short‑term memory (conversation buffer)
Store last N messages to keep context.

In [ ]:
class ShortTermMemory:
    def __init__(self, max_messages=10):
        self.messages = []
        self.max_messages = max_messages
    
    def add(self, role: str, content: str):
        self.messages.append({"role": role, "content": content})
        if len(self.messages) > self.max_messages:
            self.messages = self.messages[-self.max_messages:]
    
    def get(self) -> List[Dict]:
        return self.messages

short_mem = ShortTermMemory(max_messages=6)
short_mem.add("user", "Hi, my name is Alice.")
short_mem.add("assistant", "Hello Alice! How can I help?")
print(short_mem.get())

## 2. Long‑term memory (semantic – FAISS + embeddings)
Store user facts or previous interactions as vectors and retrieve relevant ones on demand.

In [ ]:
class LongTermMemory:
    def __init__(self, embedding_model="text-embedding-3-small"):
        self.embedding_model = embedding_model
        self.facts = []  # list of strings
        self.index = None
        self.dim = None
    
    def _embed(self, texts: List[str]) -> np.ndarray:
        response = client.embeddings.create(
            model=self.embedding_model,
            input=texts
        )
        return np.array([item.embedding for item in response.data]).astype('float32')
    
    def add_fact(self, fact: str):
        self.facts.append(fact)
        self._rebuild_index()
    
    def _rebuild_index(self):
        if not self.facts:
            return
        embeddings = self._embed(self.facts)
        self.dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(self.dim)
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings)
    
    def retrieve(self, query: str, k=2) -> List[str]:
        if not self.facts or self.index is None:
            return []
        q_emb = self._embed([query])
        faiss.normalize_L2(q_emb)
        scores, indices = self.index.search(q_emb, k)
        return [self.facts[i] for i in indices[0] if i < len(self.facts)]

long_mem = LongTermMemory()
long_mem.add_fact("User's name is Alice.")
long_mem.add_fact("Alice loves machine learning.")
long_mem.add_fact("Alice lives in Paris.")

retrieved = long_mem.retrieve("Where does the user live?")
print("Retrieved facts:", retrieved)

## 3. Combined memory agent
Use short‑term for recent conversation, long‑term for personal facts.

In [ ]:
def ask_with_memory(user_input: str) -> str:
    # Retrieve long‑term facts relevant to the input
    facts = long_mem.retrieve(user_input, k=2)
    context = ""
    if facts:
        context = "Relevant facts about the user:\n" + "\n".join(f"- {f}" for f in facts)
    
    # Build messages: short‑term history + context + new input
    messages = short_mem.get().copy()
    if context:
        messages.append({"role": "system", "content": context})
    messages.append({"role": "user", "content": user_input})
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages
    )
    answer = response.choices[0].message.content
    
    # Update short‑term memory
    short_mem.add("user", user_input)
    short_mem.add("assistant", answer)
    
    # Optionally extract new facts for long‑term memory (simplified: ask LLM to summarise)
    if "remember that" in user_input.lower() or "my name is" in user_input.lower():
        long_mem.add_fact(user_input)
    
    return answer

print(ask_with_memory("My favorite color is blue."))
print(ask_with_memory("What is my favorite color?"))

## 4. View memory contents

In [ ]:
print("Short‑term:", short_mem.get())
print("Long‑term facts:", long_mem.facts)